In [347]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Feature engineering notebook ready!")

Feature engineering notebook ready!


In [348]:
from pathlib import Path

project_root = Path.cwd().parent

data_file = project_root / "data" / "processed" / "case_1_1min.csv"

df = pd.read_csv(data_file)

df.head()

,minute,heart_rate,spo2,respiratory_rate,mean_arterial_pressure,spo2_valid_count,samples_in_minute,spo2_coverage
0,0,79.0,97.0,NaN,-9.0,29,30,0.966667
1,1,85.0,97.0,NaN,-9.0,30,30,1.000000
2,2,88.0,97.0,NaN,-9.0,30,30,1.000000
3,3,87.0,98.0,NaN,-9.0,30,30,1.000000
4,4,86.0,100.0,NaN,-9.0,30,30,1.000000


In [349]:
print("Shape:", df.shape)
print()
print(df.columns)

Shape: (193, 8)

Index(['minute', 'heart_rate', 'spo2', 'respiratory_rate',
       'mean_arterial_pressure', 'spo2_valid_count', 'samples_in_minute',
       'spo2_coverage'],
      dtype='str')


In [350]:
df_model = df.copy()

df_model.loc[
    df_model["heart_rate"] <= 0,
    "heart_rate"
] = np.nan

df_model.loc[
    (df_model["spo2"] < 0) |
    (df_model["spo2"] > 100),
    "spo2"
] = np.nan

df_model.loc[
    df_model["respiratory_rate"] < 0,
    "respiratory_rate"
] = np.nan

df_model.loc[
    df_model["mean_arterial_pressure"] <= 0,
    "mean_arterial_pressure"
] = np.nan

In [351]:
df_model["future_spo2_5min"] = df_model["spo2"].shift(-5)

In [352]:
df_model["spo2_lag_1"] = df_model["spo2"].shift(1)
df_model["spo2_lag_3"] = df_model["spo2"].shift(3)
df_model["spo2_lag_5"] = df_model["spo2"].shift(5)

In [353]:
df_model[
    [
        "minute",
        "spo2_lag_5",
        "spo2_lag_3",
        "spo2_lag_1",
        "spo2",
        "future_spo2_5min"
    ]
].head(12)

,minute,spo2_lag_5,spo2_lag_3,spo2_lag_1,spo2,future_spo2_5min
0,0,NaN,NaN,NaN,97.0,100.0
1,1,NaN,NaN,97.0,97.0,100.0
2,2,NaN,NaN,97.0,97.0,100.0
3,3,NaN,97.0,97.0,98.0,100.0
4,4,NaN,97.0,98.0,100.0,100.0
5,5,97.0,97.0,100.0,100.0,100.0
6,6,97.0,98.0,100.0,100.0,100.0
7,7,97.0,100.0,100.0,100.0,100.0
8,8,98.0,100.0,100.0,100.0,100.0
9,9,100.0,100.0,100.0,100.0,100.0


In [354]:
df_model["spo2_change_1min"] = (
    df_model["spo2"] -
    df_model["spo2_lag_1"]
)

df_model["spo2_change_3min"] = (
    df_model["spo2"] -
    df_model["spo2_lag_3"]
)

In [355]:
df_model["spo2_rolling_mean_5"] = (
    df_model["spo2"]
    .rolling(window=5, min_periods=3)
    .mean()
)

In [356]:
df_model["spo2_rolling_std_5"] = (
    df_model["spo2"]
    .rolling(window=5, min_periods=3)
    .std()
)

In [357]:
df_model["heart_rate_lag_1"] = (
    df_model["heart_rate"].shift(1)
)

df_model["heart_rate_rolling_mean_5"] = (
    df_model["heart_rate"]
    .rolling(window=5, min_periods=3)
    .mean()
)

In [358]:
df_model["map_lag_1"] = (
    df_model["mean_arterial_pressure"].shift(1)
)

df_model["map_rolling_mean_5"] = (
    df_model["mean_arterial_pressure"]
    .rolling(window=5, min_periods=3)
    .mean()
)

In [359]:
df_model["rr_lag_1"] = (
    df_model["respiratory_rate"].shift(1)
)

df_model["rr_rolling_mean_5"] = (
    df_model["respiratory_rate"]
    .rolling(window=5, min_periods=3)
    .mean()
)

In [360]:
feature_columns = [
    "heart_rate",
    "respiratory_rate",
    "mean_arterial_pressure",
    "spo2",

    "spo2_lag_1",
    "spo2_lag_3",
    "spo2_lag_5",

    "spo2_change_1min",
    "spo2_change_3min",

    "spo2_rolling_mean_5",
    "spo2_rolling_std_5",

    "heart_rate_lag_1",
    "heart_rate_rolling_mean_5",

    "map_lag_1",
    "map_rolling_mean_5",

    "rr_lag_1",
    "rr_rolling_mean_5"
]

df_model[
    feature_columns + ["future_spo2_5min"]
].head(15)

,heart_rate,respiratory_rate,mean_arterial_pressure,spo2,spo2_lag_1,spo2_lag_3,spo2_lag_5,spo2_change_1min,spo2_change_3min,spo2_rolling_mean_5,spo2_rolling_std_5,heart_rate_lag_1,heart_rate_rolling_mean_5,map_lag_1,map_rolling_mean_5,rr_lag_1,rr_rolling_mean_5,future_spo2_5min
0,79.0,NaN,NaN,97.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0
1,85.0,NaN,NaN,97.0,97.0,NaN,NaN,0.0,NaN,NaN,NaN,79.0,NaN,NaN,NaN,NaN,NaN,100.0
2,88.0,NaN,NaN,97.0,97.0,NaN,NaN,0.0,NaN,97.00,0.000000,85.0,84.00,NaN,NaN,NaN,NaN,100.0
3,87.0,NaN,NaN,98.0,97.0,97.0,NaN,1.0,1.0,97.25,0.500000,88.0,84.75,NaN,NaN,NaN,NaN,100.0
4,86.0,NaN,NaN,100.0,98.0,97.0,NaN,2.0,3.0,97.80,1.303840,87.0,85.00,NaN,NaN,NaN,NaN,100.0
5,80.0,NaN,NaN,100.0,100.0,97.0,97.0,0.0,3.0,98.40,1.516575,86.0,85.20,NaN,NaN,NaN,NaN,100.0
6,78.0,NaN,NaN,100.0,100.0,98.0,97.0,0.0,2.0,99.00,1.414214,80.0,83.80,NaN,NaN,NaN,NaN,100.0
7,74.0,NaN,NaN,100.0,100.0,100.0,97.0,0.0,0.0,99.60,0.894427,78.0,81.00,NaN,NaN,NaN,NaN,100.0
8,73.0,NaN,NaN,100.0,100.0,100.0,98.0,0.0,0.0,100.00,0.000000,74.0,78.20,NaN,NaN,NaN,NaN,100.0
9,120.5,9.0,NaN,100.0,100.0,100.0,100.0,0.0,0.0,100.00,0.000000,73.0,85.10,NaN,NaN,NaN,NaN,100.0


In [361]:
feature_missing = (
    df_model[
        feature_columns + ["future_spo2_5min"]
    ]
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

feature_missing

respiratory_rate             26
rr_lag_1                     26
rr_rolling_mean_5            26
map_rolling_mean_5           22
mean_arterial_pressure       22
map_lag_1                    22
future_spo2_5min             14
spo2_change_3min             12
heart_rate                   10
spo2_change_1min             10
heart_rate_rolling_mean_5    10
heart_rate_lag_1             10
spo2_rolling_mean_5           9
spo2_lag_3                    9
spo2                          9
spo2_lag_5                    9
spo2_lag_1                    9
spo2_rolling_std_5            9
dtype: int64

In [362]:
df_model["feature"] = df_model["spo2"].shift(-1)

In [363]:
df_model = df_model.drop(columns=["feature"])

In [364]:
"feature" in df_model.columns

False

In [365]:
feature_missing = (
    df_model[
        feature_columns + ["future_spo2_5min"]
    ]
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

feature_missing.to_frame("missing_count")

,missing_count
respiratory_rate,26
rr_lag_1,26
rr_rolling_mean_5,26
map_rolling_mean_5,22
mean_arterial_pressure,22
map_lag_1,22
future_spo2_5min,14
spo2_change_3min,12
heart_rate,10
spo2_change_1min,10


In [366]:
feature_missing_report = pd.DataFrame({
    "missing_count": df_model[
        feature_columns + ["future_spo2_5min"]
    ].isnull().sum(),

    "missing_percent": (
        df_model[
            feature_columns + ["future_spo2_5min"]
        ].isnull().mean() * 100
    ).round(2)
})

feature_missing_report.sort_values(
    "missing_percent",
    ascending=False
)

,missing_count,missing_percent
respiratory_rate,26,13.47
rr_lag_1,26,13.47
rr_rolling_mean_5,26,13.47
map_rolling_mean_5,22,11.40
mean_arterial_pressure,22,11.40
map_lag_1,22,11.40
future_spo2_5min,14,7.25
spo2_change_3min,12,6.22
heart_rate,10,5.18
spo2_change_1min,10,5.18


In [367]:
spo2_features = [
    "spo2",
    "spo2_lag_1",
    "spo2_lag_3",
    "spo2_lag_5",
    "spo2_change_1min",
    "spo2_change_3min",
    "spo2_rolling_mean_5",
    "spo2_rolling_std_5"
]

In [368]:
spo2_hr_features = spo2_features + [
    "heart_rate",
    "heart_rate_lag_1",
    "heart_rate_rolling_mean_5"
]

In [369]:
all_vital_features = spo2_hr_features + [
    "respiratory_rate",
    "rr_lag_1",
    "rr_rolling_mean_5",
    "mean_arterial_pressure",
    "map_lag_1",
    "map_rolling_mean_5"
]

In [370]:
def count_complete_rows(features):
    required_columns = features + ["future_spo2_5min"]

    return (
        df_model[required_columns]
        .dropna()
        .shape[0]
    )

In [371]:
print(
    "SpO2-only usable rows:",
    count_complete_rows(spo2_features)
)

print(
    "SpO2 + HR usable rows:",
    count_complete_rows(spo2_hr_features)
)

print(
    "All vitals usable rows:",
    count_complete_rows(all_vital_features)
)

SpO2-only usable rows: 174
SpO2 + HR usable rows: 174
All vitals usable rows: 163


In [ ]:
feature_file = (
    project_root /
    "data" /
    "processed" /
    "case_1_features.csv"
)

df_model.to_csv(
    feature_file,
    index=False
)

print("Feature dataset saved to:")
print(feature_file)

Feature dataset saved to:
c:\Users\samat\OneDrive\Documents\oxygen-saturation-prediction\data\processed\case_1_features.csv


: 